# Simulate k-TSP path (a.k.a. k-cardinality TSP path) / orienteering-family problem

In [1]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import numpy as np

def solve_k_tsp_path_ortools(D, start, end, K, time_limit=10):
    """
    Fixed start/end, simple path, visit exactly K nodes total (including start and end).
    Uses OR-Tools Routing (C++ backend).
    """
    N = D.shape[0]
    if not (2 <= K <= N):
        raise ValueError(f"K must be in [2, N]. Got K={K}, N={N}.")
    if start == end and K > 1:
        raise ValueError("start and end must be different for an open path with K>=2.")

    manager = pywrapcp.RoutingIndexManager(N, 1, [start], [end])
    routing = pywrapcp.RoutingModel(manager)

    # Distance callback (must be int for routing)
    def dist_cb(i, j):
        return int(D[manager.IndexToNode(i), manager.IndexToNode(j)])

    dist_idx = routing.RegisterTransitCallback(dist_cb)
    routing.SetArcCostEvaluatorOfAllVehicles(dist_idx)

    # Make all non-(start,end) nodes optional with a big penalty for skipping
    SKIP_PENALTY = 10**9
    for node in range(N):
        if node not in (start, end):
            routing.AddDisjunction([manager.NodeToIndex(node)], SKIP_PENALTY)  # positional args

    # Count dimension: each traversed arc adds 1
    def one_cb(i, j):
        return 1

    one_idx = routing.RegisterTransitCallback(one_cb)
    routing.AddDimension(
        one_idx,
        slack_max=0,
        capacity=K - 1,                # max arcs = K-1
        fix_start_cumul_to_zero=True,
        name="count"
    )
    count_dim = routing.GetDimensionOrDie("count")

    # Enforce exactly K nodes visited <=> exactly K-1 arcs traversed
    count_dim.CumulVar(routing.End(0)).SetValue(K - 1)

    # Search
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.time_limit.seconds = int(time_limit)
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH

    sol = routing.SolveWithParameters(params)
    if sol is None:
        return None, None

    # Extract path + compute cost
    path = []
    idx = routing.Start(0)
    total_cost = 0
    visited = set()

    while not routing.IsEnd(idx):
        node = manager.IndexToNode(idx)
        path.append(node)
        visited.add(node)
        nxt = sol.Value(routing.NextVar(idx))
        total_cost += routing.GetArcCostForVehicle(idx, nxt, 0)
        idx = nxt

    path.append(manager.IndexToNode(idx))  # end
    visited.add(path[-1])

    # sanity: should be exactly K unique nodes
    if len(visited) != K:
        # This should not happen with the constraint, but it's a helpful guardrail.
        raise RuntimeError(f"Expected K={K} unique visited nodes, got {len(visited)}. Path: {path}")

    return path, total_cost

def simulate_problem(N=30, K=10, seed=0):
    rng = np.random.default_rng(seed)
    coords = rng.uniform(0, 100, size=(N,2))

    D = np.sqrt(((coords[:,None,:] - coords[None,:,:])**2).sum(-1))
    start, end = 0, N-1

    return D, coords, start, end, K

import matplotlib.pyplot as plt
import numpy as np

def plot_k_tsp_path(coords, path, start, end, ax=None):
    """
    Visualize a k-TSP path in 2D with arrows.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))

    coords = np.asarray(coords)
    path = np.asarray(path)

    # All points (background)
    ax.scatter(coords[:, 0], coords[:, 1],
               c="lightgray", s=40, label="Unused nodes", zorder=1)

    # Path nodes
    ax.scatter(coords[path, 0], coords[path, 1],
               c="tab:blue", s=80, label="Visited nodes", zorder=3)

    # Start / end
    ax.scatter(coords[start, 0], coords[start, 1],
               c="green", s=120, marker="o", label="Start", zorder=4)
    ax.scatter(coords[end, 0], coords[end, 1],
               c="red", s=120, marker="X", label="End", zorder=4)

    # Draw arrows
    for i in range(len(path) - 1):
        x0, y0 = coords[path[i]]
        x1, y1 = coords[path[i + 1]]

        ax.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(
                arrowstyle="->",
                color="tab:blue",
                lw=2,
                shrinkA=8,
                shrinkB=8
            ),
            zorder=2
        )

        # Optional: label order
        ax.text(x0, y0, str(i), fontsize=9,
                ha="center", va="center",
                color="black", zorder=5)

    ax.text(coords[path[-1], 0], coords[path[-1], 1],
            str(len(path) - 1), fontsize=9,
            ha="center", va="center", color="black", zorder=5)

    ax.set_title(f"k-TSP Path (K={len(path)})")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal", adjustable="box")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)

    return ax


ModuleNotFoundError: No module named 'ortools'

In [2]:
D, coords, start, end, K = simulate_problem(N=15, K=15, seed=0)

path_ort, cost_ort = solve_k_tsp_path_ortools(D, start, end, K)

print("Path:", path_ort, '\nCost:', cost_ort)

plot_k_tsp_path(coords, path_ort, start, end)
plt.show()


NameError: name 'simulate_problem' is not defined